# bebe-ia sur Colab — l'entrainement long

Le bebe apprend **en ligne, sans supervision de confiance**.
Objectif de ce notebook : des runs **1M, 10M interactions, 5 seeds**,
synchronisees avec le fragment GitHub via `scripts/synchroniser.py`.

Cellules : Drive -> clone -> install -> config -> donnees (synthetique, pool 1200)
-> boucle (GPU amp, tqdm) -> evaluation -> courbes -> checkpoints -> reprise.
**Reproductible : meme seed = memes resultats.**

Plan A (50k, validé) : ECE ~0.02 / acc ~0.58 — pipeline Colab innocent.
Etape en cours : **1M** (le vrai run, source synthetique, heldout items).

In [ ]:
# 1. Drive — cache + checkpoints SUR LE COMPTE mathieumaire77@gmail.com
#    Le Drive monte = compte du RUNTIME Colab. La cellule VERIFIE le compte
#    et BLOQUE si ce n'est pas mathieumaire77 (changer compte en haut, re-run).
from google.colab import drive
drive.mount('/content/drive')
import os, subprocess
try:
    compte = subprocess.run(['gcloud', 'auth', 'list', '--filter=status:active', '--format=value(account)'],
                            capture_output=True, text=True, timeout=15).stdout.strip()
except Exception:
    compte = ''
print('COMPTE RUNTIME :', compte or 'inconnu — VERIFIE le menu compte en haut a droite')
if compte and 'mathieumaire77' not in compte:
    raise SystemExit('MAUVAIS COMPTE (' + compte + ') : ce run doit tourner sous '
                     'mathieumaire77@gmail.com. Change de compte Colab, '
                     'Disconnect and delete runtime, puis relance ce notebook.')
CACHE = '/content/drive/MyDrive/bebe_ia_cache'
os.makedirs(CACHE, exist_ok=True)
print('cache :', CACHE)

In [ ]:
# 2. Clone + install
%cd /content
!git clone https://github.com/ehwaza/bebe-ia.git
%cd /content/bebe-ia
!pip install -q -r requirements.txt tqdm
!ls

In [ ]:
# 3. Imports + config (SEEDS TRACEES — meme seed = memes resultats)
import sys, json, time
sys.path.insert(0, '/content/bebe-ia')
sys.path.insert(0, '/content/bebe-ia/scripts')
import torch
from entrainer_local import boucle_entrainement

CONFIG = {
    'n': 1_000_000,            # ETAPE 1M (plan A 50k = valide : ECE ~0.02, acc ~0.58)
    'seeds': [1234, 1235, 1236, 1237, 1238],
    'source': 'synthetique',    # items pool 1200 = MEME source que heldout.npz
    'cache': CACHE,
    'eval_every': 5000,         # 200 points de courbe sur 1M
    'ckpt_every': 100_000,      # anti-deconnexion (~10 min entre ckpt)
    'amp': False,               # numerique exacte pour comparer au local
}
print(json.dumps({k: v for k, v in CONFIG.items() if k != 'cache'}, indent=2))
print('GPU :', torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'CPU')

In [ ]:
# 4. Donnees — synthetique : pool fini de 1200 items (le monde revient)
#    (source = heldout : train et eval sont sur le MEME graphe)
from entrainer_local import charger_donnees
SEED_DATA = CONFIG['seeds'][0] * 7919 + 13
gen = charger_donnees(CONFIG['source'], seed_data=SEED_DATA, cache=CONFIG['cache'])
for _ in range(3):
    src, cible = next(gen)
    print(repr(src), '->', repr(cible))

In [ ]:
# 5. Boucle d'entrainement — RUN 1M (dossier NEUF : courbe complete 0 -> 1M)
#    La reprise : relancer cette cellule, le dernier checkpoint est repris
#    via --reprendre (parent_dir) — n compte les interactions NOUVELLES.
from tqdm.notebook import tqdm

SEED = CONFIG['seeds'][0]        # <- change de seed pour chaque run
SEED_DATA = SEED * 7919 + 13     # unique par run, tracee dans meta
OUT = f'/content/drive/MyDrive/bebe_runs/synth1M_seed_{SEED}'
#    ^ dossier NEUF : planA_synth_seed_1234 = le run de validation 50k, on n'y
#    touche pas ; ce run repart de 0 pour une courbe 0 -> 1M propre.

resume = boucle_entrainement(
    n=CONFIG['n'], out_dir=OUT, seed=SEED,
    seed_data=SEED * 7919 + 13,          # unique par run, tracee dans meta
    source=CONFIG['source'], cache=CONFIG['cache'],
    heldout_path='/content/bebe-ia/heldout.npz',
    eval_every=CONFIG['eval_every'], ckpt_every=CONFIG['ckpt_every'],
    amp=CONFIG['amp'],
    parent_dir=OUT + '/etat')             # reprise si le paquet existe
print(json.dumps(resume, indent=2))

In [ ]:
# 6. Evaluation + courbes (ECE, accuracy, risk-coverage)
import numpy as np, matplotlib.pyplot as plt
from bebe.metriques import resume_metriques

z = np.load(OUT + '/hist.npz')
conf, ok, ece = z['conf'], z['ok'], z['ece']
print(json.dumps(resume_metriques(conf, ok), indent=2))

fig, axs = plt.subplots(1, 2, figsize=(14, 4))
axs[0].plot(ece); axs[0].set_xlabel('evaluation x CONFIG["eval_every"]')
axs[0].set_ylabel('ECE held-out'); axs[0].set_title('Calibration')
axs[1].plot(np.cumsum(ok) / np.arange(1, len(ok) + 1))
axs[1].set_xlabel('interactions'); axs[1].set_ylabel('accuracy cumulee')
axs[1].set_title('Apprentissage')
for ax in axs: ax.grid(alpha=0.3)
plt.tight_layout(); plt.savefig(OUT + '/courbes.png', dpi=120); plt.show()

In [ ]:
# 7. Synchronisation avec le fragment GitHub (fusion par confiance)
#    A lancer depuis le clone bebe-ia, avec les etats locaux copies dans etats/.
!cp -r {OUT}/etat /content/bebe-ia/etats/local
!cd /content/bebe-ia && git pull --rebase --autostash || true
!cd /content/bebe-ia && python scripts/synchroniser.py
# Pour uploader (necessite un token avec droits d'ecriture) :
# !cd /content/bebe-ia && python scripts/synchroniser.py --push